In [1]:
import torch
import pandas as pd
from epiweeks import Week
import preprocess_data as prep
import model as mc
import matplotlib.pyplot as plt 
from loss_func import WeightedWISLossFromDistribution, LogNormalCRPSLoss

pd.options.mode.chained_assignment = None

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

regioes_estados = {
        'Sul': ['SC', 'PR', 'RS'],
        'Sudeste': ['SP', 'MG', 'RJ', 'ES'],
        'Nordeste': ['BA', 'CE', 'PE', 'PB', 'PI', 'RN', 'MA', 'AL', 'SE'],
        'Centro-Oeste': ['DF', 'MT', 'MS', 'GO'],
        'Norte': ['RO', 'AC', 'AM', 'RR', 'PA', 'AP', 'TO']
    } 
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

columns_to_normalize = ['casos','epiweek', 'biome', 'enso']

predict_n = 26
max_epiweek = 26
    
boxcox = False

TEST_YEAR = 2026


In [2]:
device

device(type='cuda')

In [2]:
batch_size = 2
epochs = 200
media =True
verbose = 1
doenca = 'dengue'
min_delta = 0.002
patience= 25
lr = 2e-4
min_year = 2015
model_name = f'enso_{max_epiweek}_{TEST_YEAR}_V3'

In [3]:
df = prep.load_cases_data(filename= f'./data/{doenca}.csv.gz')

enso = prep.load_enso_weekly(filename='data/enso_weekly_forecast_up_05_08.csv')

enso_neutro = prep.load_enso_weekly(filename='data/enso_weekly_neutro.csv')

In [ ]:
for region in regioes_estados.keys(): 

    print(region)

    label = f'{region}_{TEST_YEAR-1}_{model_name}'

    df_reg = df.loc[df.uf.isin(regioes_estados[region])]
    df_reg = df_reg.loc[df_reg.index >= pd.to_datetime(Week(2015,1).startdate())]


    X_train, y_train, X_future, norm_values, norm_enso =  prep.generate_regional_train_samples(df_reg, enso, 
                                                                                    TEST_YEAR,
                                                                                    max_epiweek = max_epiweek,
                                                                                    columns_to_normalize = columns_to_normalize,
                                                                                    min_year = min_year, 
                                                                                    boxcox = boxcox,
                                                                                    media = media)


    model = mc. LSTMWithFutureCovariatesV3(

                    hidden=64,

                    past_features=5,

                    future_cov_size=predict_n,

                    predict_n=predict_n,

                    dropout=0.2
                        )
    
    #model_path = f'./saved_models/trained_dengue_{region}_{TEST_YEAR-1}_enso_media.pt'
    #model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    #model.to(device)  
    
    trained_model, hist = mc.train_model(

            model=model,

            X_train=X_train,

            X_future=X_future,

            Y_train=y_train,

            label=label,

            batch_size=batch_size,

            epochs=epochs,

            lr=lr,

            patience=patience,

            verbose=verbose,

            doenca=doenca,

            #criterion =   LogNormalCRPSLoss().to(device)
)


Sul
torch.Size([327, 78, 5])
